In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
from pyspark.sql import functions as F

SRC = "/Volumes/workspace/olist_raw/landing"
tables = ["customers","geolocation","order_items","order_payments",
          "order_reviews","orders","products","sellers",
          "product_category_name_translation"]

for t in tables:
    df = spark.read.parquet(f"{SRC}/{t}.parquet")
    (df.withColumn("_ingested_at", F.current_timestamp())
       .withColumn("_source_file", F.lit(f"{t}.parquet"))
       .write.format("delta").mode("overwrite")
       .saveAsTable(f"workspace.bronze.{t}"))
    print(t, df.count())

customers 99441
geolocation 1000163
order_items 112650
order_payments 103886
order_reviews 99224
orders 99441
products 32951
sellers 3095
product_category_name_translation 71


In [0]:
from pyspark.sql import functions as F

class DQError(Exception):
    pass

def dq_check(table, pk=None, not_null=None, expected_cols=None,
             min_rows=1, fks=None):
    """
    pk: list of columns forming the primary key (uniqueness check)
    not_null: columns that must have no nulls
    expected_cols: columns that must exist (schema validation)
    fks: list of (col, ref_table, ref_col) for referential integrity
    """
    df = spark.table(table)
    results = []

    # row count
    n = df.count()
    results.append(("row_count>=%d" % min_rows, n >= min_rows, n))

    # schema validation
    if expected_cols:
        missing = [c for c in expected_cols if c not in df.columns]
        results.append(("schema_has_expected_cols", not missing, missing))

    # null checks
    for c in (not_null or []):
        nulls = df.filter(F.col(c).isNull()).count()
        results.append((f"not_null:{c}", nulls == 0, nulls))

    # uniqueness
    if pk:
        dups = n - df.select(*pk).distinct().count()
        results.append((f"unique:{'+'.join(pk)}", dups == 0, dups))

    # referential integrity
    for col, ref_table, ref_col in (fks or []):
        ref = spark.table(ref_table).select(F.col(ref_col).alias("_k")).distinct()
        orphans = (df.select(col).where(F.col(col).isNotNull()).distinct()
                     .join(ref, F.col(col) == F.col("_k"), "left_anti").count())
        results.append((f"fk:{col}->{ref_table}.{ref_col}", orphans == 0, orphans))

    failed = [r for r in results if not r[1]]
    print(f"--- DQ {table} ---")
    for name, ok, val in results:
        print(("PASS" if ok else "FAIL"), name, val)
    if failed:
        raise DQError(f"{table}: {len(failed)} check(s) failed")
    return True

In [0]:
dq_check("workspace.bronze.orders", min_rows=90000,
         expected_cols=["order_id","customer_id","order_status","order_purchase_timestamp"])
dq_check("workspace.bronze.customers", min_rows=90000, expected_cols=["customer_id","customer_unique_id"])
dq_check("workspace.bronze.products", min_rows=30000, expected_cols=["product_id","product_category_name"])

--- DQ workspace.bronze.orders ---
PASS row_count>=90000 99441
PASS schema_has_expected_cols []
--- DQ workspace.bronze.customers ---
PASS row_count>=90000 99441
PASS schema_has_expected_cols []
--- DQ workspace.bronze.products ---
PASS row_count>=30000 32951
PASS schema_has_expected_cols []


True